# Этап 4. Предобработка данных

**Цель этапа:** подготовить данные для обучения моделей машинного обучения.

**Шаги:**
1. Загрузка размеченного датасета
2. Кодирование текстовых признаков (One-Hot Encoding)
3. Разделение на три выборки: Train (60%) / Test (20%)
4. Масштабирование числовых признаков (StandardScaler) 

## 1. Загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

df = pd.read_csv('../data/russian_food_labeled.csv')

print('Размер датасета:', df.shape)
print()
print('Столбцы:', df.columns.tolist())
print()
print('Целевые переменные:')
targets = ['target_weightloss', 'target_gainmass', 'target_balance', 'target_sugar']
labels  = ['Похудение', 'Набор массы', 'Баланс/ЗОЖ', 'Контроль сахара']
for t, l in zip(targets, labels):
    n1 = df[t].sum()
    n0 = len(df) - n1
    print(f'  {l}: нужна замена={n1}, не нужна={n0}')

Размер датасета: (400, 14)

Столбцы: ['Food_Item', 'Category', 'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fat (g)', 'Fiber (g)', 'Sugars (g)', 'Sodium (mg)', 'Meal_Type', 'target_weightloss', 'target_gainmass', 'target_balance', 'target_sugar']

Целевые переменные:
  Похудение: нужна замена=174, не нужна=226
  Набор массы: нужна замена=179, не нужна=221
  Баланс/ЗОЖ: нужна замена=152, не нужна=248
  Контроль сахара: нужна замена=138, не нужна=262


## 2. Кодирование текстовых признаков (One-Hot Encoding)

Текстовые столбцы `Category` и `Meal_Type` нужно превратить в числовые.
**One-Hot Encoding** — каждое уникальное значение становится отдельным столбцом из 0 и 1.


In [2]:
# Применяем One-Hot Encoding к категориальным столбцам
df_encoded = pd.get_dummies(df, columns=['Category', 'Meal_Type'], drop_first=False)

# Убираем нечисловые и целевые столбцы для признаков
feature_cols = [c for c in df_encoded.columns
                if c not in ['Food_Item'] + targets
                and not c.startswith('target_')]

numeric_features = ['Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)',
                     'Fat (g)', 'Fiber (g)', 'Sugars (g)', 'Sodium (mg)']

print(f'Признаков до кодирования: {len(numeric_features)} (числовые)')
print(f'Признаков после кодирования: {len(feature_cols)}')
print()
print('Новые столбцы после One-Hot Encoding:')
new_cols = [c for c in feature_cols if 'Category_' in c or 'Meal_Type_' in c]
for c in new_cols:
    print(f'  {c}')

Признаков до кодирования: 7 (числовые)
Признаков после кодирования: 27

Новые столбцы после One-Hot Encoding:
  Category_Бобовые
  Category_Выпечка
  Category_Гарнир
  Category_Десерт
  Category_Европейское
  Category_Завтрак
  Category_Крупа
  Category_Молочное
  Category_Мясо
  Category_Напиток
  Category_Овощ
  Category_Орехи
  Category_Рыба
  Category_Салат
  Category_Суп
  Category_Фрукт
  Category_Яйца
  Meal_Type_Завтрак
  Meal_Type_Обед
  Meal_Type_Перекус


## 3. Разделение на две выборки: Train / Test

- **Train (80%)**
- **Test (20%)** 

Один общий split для всех 4 целей. Стратификация — по `target_balance` (при проверке дала
наименьший разброс баланса классов сразу по всем 4 целям).

In [3]:
X_all = df_encoded[feature_cols].copy()

# Единый split индексов — стратификация по target_balance 
idx_train, idx_test = train_test_split(
    df.index.values, 
    test_size=0.2,          
    random_state=42, 
    stratify=df['target_balance']  
)

X_train = X_all.loc[idx_train].copy()
X_test  = X_all.loc[idx_test].copy()

y_train = {label: df.loc[idx_train, t].values for label, t in zip(labels, targets)}
y_test  = {label: df.loc[idx_test,  t].values for label, t in zip(labels, targets)}

print('Размеры выборок:')
total = len(df)
print('  Train:     ', len(X_train), 'строк (', round(len(X_train)/total*100), '%)')
print('  Test:      ', len(X_test),  'строк (', round(len(X_test)/total*100),  '%)')
print('  Итого:     ', len(X_train)  + len(X_test), 'строк')

Размеры выборок:
  Train:      320 строк ( 80 %)
  Test:       80 строк ( 20 %)
  Итого:      400 строк


## 4. Масштабирование числовых признаков (StandardScaler)


In [4]:
scaler = StandardScaler()

# fit — только на train, transform — на всех трёх выборках
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features]  = scaler.transform(X_test[numeric_features])

print('Числовые признаки после масштабирования (Train):')
print(X_train[numeric_features].describe().round(2))

Числовые признаки после масштабирования (Train):
       Calories (kcal)  Protein (g)  Carbohydrates (g)  Fat (g)  Fiber (g)  \
count           320.00       320.00             320.00   320.00     320.00   
mean              0.00        -0.00               0.00    -0.00      -0.00   
std               1.00         1.00               1.00     1.00       1.00   
min              -1.36        -1.15              -1.06    -0.78      -0.87   
25%              -0.78        -0.80              -0.74    -0.63      -0.72   
50%              -0.25        -0.39              -0.36    -0.40      -0.09   
75%               0.57         0.71               0.50     0.36       0.44   
max               4.28         3.12               4.25     6.29       5.71   

       Sugars (g)  Sodium (mg)  
count      320.00       320.00  
mean         0.00         0.00  
std          1.00         1.00  
min         -0.64        -0.74  
25%         -0.52        -0.58  
50%         -0.35        -0.20  
75%         -0.04

## 5. Проверка баланса классов в выборках

In [5]:
print('Баланс классов (доля "нужна замена = 1") по выборкам:')
print(f'{"Цель":<20} {"Train":>10} {"Test":>10}')
print('-' * 60)
for label in labels:
    train_pct = y_train[label].mean() * 100
    test_pct  = y_test[label].mean()  * 100
    print(f'{label:<20} {train_pct:>9.1f}% {test_pct:>9.1f}%')

Баланс классов (доля "нужна замена = 1") по выборкам:
Цель                      Train       Test
------------------------------------------------------------
Похудение                 42.8%      46.2%
Набор массы               45.0%      43.8%
Баланс/ЗОЖ                38.1%      37.5%
Контроль сахара           34.7%      33.8%


## 7. Сохранение результатов предобработки

In [6]:
# Сохраняем splits и scaler для использования в следующих этапах
os.makedirs('../data', exist_ok=True)

splits = {
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
}

with open('../data/splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../data/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print('Сохранено:')
print('  ../data/splits.pkl  — X_train//X_test + y_train/y_test (по 4 целям)')
print('  ../data/scaler.pkl  — обученный StandardScaler (fit только на train)')
print('  ../data/feature_cols.pkl — список признаков')

Сохранено:
  ../data/splits.pkl  — X_train//X_test + y_train/y_test (по 4 целям)
  ../data/scaler.pkl  — обученный StandardScaler (fit только на train)
  ../data/feature_cols.pkl — список признаков


## 8. Выводы по Этапу 4 


1. Текстовые признаки `Category` (17 значений) и `Meal_Type` (3 значения) закодированы через One-Hot Encoding — итого получилось **27** признаков вместо исходных 7 числовых.
2. Данные разделены на **две** выборки: **Train 80% / Test 20%**, один общий split для всех 4 целей (стратификация по `target_balance`), с фиксированным `random_state=42` для воспроизводимости.
3. Масштабирование (StandardScaler) выполнено **после** split и обучено **только на Train** — так избегаем утечки данных из test-выборки.
4. Баланс классов по всем 4 целям сохраняется в пределах ~1-4 п.п. между Train/Test.
5. Все результаты сохранены в `.pkl` файлы для использования на этапе обучения моделей.